In [2]:
import pandas as pd
import requests

WARSAW_LAT, WARSAW_LON = 52.2297, 21.0122

In [3]:
imgw = requests.get("https://danepubliczne.imgw.pl/api/data/synop", timeout=10).json()
imgw_df = pd.DataFrame(imgw)

print(len(imgw_df), "stations")
print(imgw_df.dtypes)
imgw_df[imgw_df["stacja"] == "Warszawa"]

62 stations
id_stacji              object
stacja                 object
data_pomiaru           object
godzina_pomiaru        object
temperatura            object
predkosc_wiatru        object
kierunek_wiatru        object
wilgotnosc_wzgledna    object
suma_opadu             object
cisnienie              object
dtype: object


,id_stacji,stacja,data_pomiaru,godzina_pomiaru,temperatura,predkosc_wiatru,kierunek_wiatru,wilgotnosc_wzgledna,suma_opadu,cisnienie
55,12375,Warszawa,2026-10-26,16,9.1,2,240,75.8,3.6,1018.4


In [4]:
params = {
    "latitude": WARSAW_LAT,
    "longitude": WARSAW_LON,
    "hourly": "temperature_2m,precipitation,rain,snowfall,weather_code,wind_speed_10m",
    "past_days": 7,
    "forecast_days": 1,
    "timezone": "UTC",
}
om = requests.get("https://api.open-meteo.com/v1/forecast", params=params, timeout=10).json()

print(om.keys())
print(om["hourly_units"])

om_df = pd.DataFrame(om["hourly"])
om_df["time"] = pd.to_datetime(om_df["time"])
print(len(om_df), "hours")
om_df.head()

dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'hourly_units', 'hourly'])
{'time': 'iso8601', 'temperature_2m': '°C', 'precipitation': 'mm', 'rain': 'mm', 'snowfall': 'cm', 'weather_code': 'wmo code', 'wind_speed_10m': 'km/h'}
192 hours


,time,temperature_2m,precipitation,rain,snowfall,weather_code,wind_speed_10m
0,2026-10-19 00:00:00,8.9,0.0,0.0,0.0,3,13.3
1,2026-10-19 01:00:00,8.6,0.0,0.0,0.0,3,13.0
2,2026-10-19 02:00:00,8.3,0.0,0.0,0.0,3,13.3
3,2026-10-19 03:00:00,8.0,0.0,0.0,0.0,3,12.2
4,2026-10-19 04:00:00,7.8,0.0,0.0,0.0,3,12.2


In [5]:
rainy = om_df[om_df["precipitation"] > 0]
print(len(rainy), "rainy hours out of", len(om_df))
rainy[["time", "precipitation", "rain", "snowfall"]]

43 rainy hours out of 192


,time,precipitation,rain,snowfall
17,2026-10-19 17:00:00,0.7,0.7,0.0
18,2026-10-19 18:00:00,0.7,0.7,0.0
19,2026-10-19 19:00:00,0.7,0.7,0.0
20,2026-10-19 20:00:00,0.2,0.2,0.0
21,2026-10-19 21:00:00,0.2,0.2,0.0
22,2026-10-19 22:00:00,0.2,0.2,0.0
23,2026-10-19 23:00:00,0.3,0.3,0.0
24,2026-10-20 00:00:00,0.2,0.2,0.0
25,2026-10-20 01:00:00,0.1,0.1,0.0
26,2026-10-20 02:00:00,0.1,0.1,0.0


In [6]:
w = imgw_df[imgw_df["stacja"] == "Warszawa"].iloc[0]
imgw_time = pd.Timestamp(f"{w['data_pomiaru']} {int(w['godzina_pomiaru']):02d}:00")

print("IMGW:", imgw_time, "temp", w["temperatura"])
print(om_df[om_df["time"] == imgw_time][["time", "temperature_2m", "precipitation"]])

IMGW: 2026-10-26 16:00:00 temp 9.1
                   time  temperature_2m  precipitation
184 2026-10-26 16:00:00            10.1            0.0
